In [0]:
# Configuration - connect to Azure Blob Storage
storage_account = "stockmarketdata2026"
storage_key = "Your Storage Key Here"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

print("Storage connected successfully")

Storage connected successfully


In [0]:
# Read raw JSON from Bronze container
from pyspark.sql.functions import col, to_timestamp, round, avg, max, min

df_bronze = spark.read.json(
    f"abfss://bronze@{storage_account}.dfs.core.windows.net/stocks/*/*.json"
)

print(f"Total records read from Bronze: {df_bronze.count()}")
df_bronze.show(5)

Total records read from Bronze: 1560
+------------------+--------------------+---------+-----------------+------------------+------------------+------------+------+-------+
|             Close|            Datetime|Dividends|             High|               Low|              Open|Stock_Splits|Symbol| Volume|
+------------------+--------------------+---------+-----------------+------------------+------------------+------------+------+-------+
| 384.7749938964844|2026-05-04 09:30:...|      0.0|386.6099853515625| 384.6099853515625| 385.6300048828125|         0.0| GOOGL|1681996|
| 387.2550048828125|2026-05-04 09:31:...|      0.0|387.3163146972656| 384.8299865722656| 384.8299865722656|         0.0| GOOGL| 163492|
| 384.8299865722656|2026-05-04 09:32:...|      0.0|387.3800048828125| 384.1300048828125|387.29998779296875|         0.0| GOOGL| 245771|
|385.20001220703125|2026-05-04 09:33:...|      0.0|            385.5|384.42999267578125| 384.7099914550781|         0.0| GOOGL|  99779|
|384.184997

In [0]:
# Bronze to Silver - clean and validate data
from pyspark.sql.functions import col, to_timestamp, round

df_silver = df_bronze \
    .filter(col("Close").isNotNull()) \
    .filter(col("Volume") > 0) \
    .withColumn("Datetime", to_timestamp(col("Datetime"))) \
    .withColumn("Close", round(col("Close"), 2)) \
    .withColumn("Open", round(col("Open"), 2)) \
    .withColumn("High", round(col("High"), 2)) \
    .withColumn("Low", round(col("Low"), 2)) \
    .select("Datetime", "Symbol", "Open", "High", "Low", "Close", "Volume")

print(f"Silver records: {df_silver.count()}")
df_silver.show(5)

# Write to Silver container as Delta
df_silver.write.format("delta") \
    .mode("overwrite") \
    .save(f"abfss://silver@{storage_account}.dfs.core.windows.net/stocks/")

print("Silver layer written successfully")

Silver records: 1560
+-------------------+------+------+------+------+------+-------+
|           Datetime|Symbol|  Open|  High|   Low| Close| Volume|
+-------------------+------+------+------+------+------+-------+
|2026-05-04 13:30:00| GOOGL|385.63|386.61|384.61|384.77|1681996|
|2026-05-04 13:31:00| GOOGL|384.83|387.32|384.83|387.26| 163492|
|2026-05-04 13:32:00| GOOGL| 387.3|387.38|384.13|384.83| 245771|
|2026-05-04 13:33:00| GOOGL|384.71| 385.5|384.43| 385.2|  99779|
|2026-05-04 13:34:00| GOOGL|384.18|385.08|383.95|384.18|2338964|
+-------------------+------+------+------+------+------+-------+
only showing top 5 rows

Silver layer written successfully


In [0]:
# Silver to Gold - aggregate for reporting
from pyspark.sql.functions import avg, max, min, sum

df_gold = df_silver \
    .groupBy("Symbol") \
    .agg(
        avg("Close").alias("avg_close"),
        max("High").alias("day_high"),
        min("Low").alias("day_low"),
        sum("Volume").alias("total_volume")
    )

print("Gold layer:")
df_gold.show()

# Write to Gold container as Delta
df_gold.write.format("delta") \
    .mode("overwrite") \
    .save(f"abfss://gold@{storage_account}.dfs.core.windows.net/stocks_summary/")

print("Gold layer written successfully")

Gold layer:
+------+------------------+--------+-------+------------+
|Symbol|         avg_close|day_high|day_low|total_volume|
+------+------------------+--------+-------+------------+
| GOOGL| 382.9281538461538|  387.38|  379.8|   107370887|
|  AAPL| 276.8278461538461|  280.63| 274.86|    76034738|
|  AMZN|271.60976923076936|   276.1| 268.86|   137070025|
|  MSFT| 415.2496923076921|  420.78| 411.98|    54250881|
+------+------------------+--------+-------+------------+

Gold layer written successfully
